# Test Web IQ MCP with the Azure OpenAI Responses API

Use an Azure OpenAI model to discover and call Web IQ's remote MCP `web` tool through Azure API Management (APIM). The request follows Microsoft's [remote MCP pattern for the Responses API](https://learn.microsoft.com/azure/foundry/openai/how-to/responses#using-remote-mcp-servers).

**Audience:** Developers validating the Web IQ MCP endpoint deployed by this lab.

**Prerequisites:**

- Complete [web-iq.ipynb](web-iq.ipynb) so the APIM endpoint and subscription exist.
- An Azure OpenAI deployment that supports the Responses API and remote MCP tools, plus the **Cognitive Services OpenAI User** role for your signed-in identity.
- Run `uv sync` at the repository root and select `.venv/bin/python` as the kernel.
- Put the connection strings described below in `labs/web-iq/.env`.

**Learning goals:** Configure the two connections without embedding secrets, invoke only the allowed `web` tool, inspect MCP response events, and confirm that APIM blocks `browse`.


## Request flow

The Azure OpenAI service—not the notebook kernel—connects to the remote MCP URL. The APIM endpoint must therefore be publicly reachable over TLS 1.2 or later. See the [Azure OpenAI Responses API MCP test flow](README.md#azure-openai-responses-api-mcp-test-flow) in the lab README.

The diagrams label Content Safety as an optional architecture step; the deployed policy always analyzes upstream response text using APIM’s system-assigned managed identity. It uses only Hate, Sexual, Violence, and SelfHarm, and adds the scores to MCP result metadata, structured content, and the first tool text block, as well as `x-content-safety-*` headers. Original tool content is preserved. Finite POST/SSE responses are buffered; the optional long-lived GET stream returns `405`.


## Outline

1. Load and validate dotenv connection strings.
2. Create the Azure OpenAI client.
3. Call the Web IQ `web` tool through APIM.
4. Inspect MCP events and assert the call succeeded.
5. Verify the APIM `browse` denial.


## 1. Configure dotenv connection strings

Add these quoted values to the existing ignored `labs/web-iq/.env` file. `Deployment` is the Azure OpenAI deployment name, not the underlying model family.

```dotenv
AZURE_OPENAI_CONNECTION_STRING="Endpoint=https://<resource>.openai.azure.com;Deployment=<deployment-name>"
WEB_IQ_MCP_CONNECTION_STRING="Endpoint=https://<apim-name>.azure-api.net/web-iq/mcp;Ocp-Apim-Subscription-Key=<apim-subscription-key>;x-apikey=<web-iq-key>"
```

> Keep `.env` local. The repository already ignores it. The notebook prints endpoints and header names only, never secret values.


In [7]:
from __future__ import annotations

import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

env_file = Path('.env') if Path('.env').exists() else Path('labs/web-iq/.env')
load_dotenv(env_file)

def parse_connection_string(variable_name: str) -> dict[str, str]:
    raw_value = os.getenv(variable_name, '').strip()
    if not raw_value:
        raise RuntimeError(f'Set {variable_name} in labs/web-iq/.env.')

    settings: dict[str, str] = {}
    for segment in raw_value.split(';'):
        if not segment.strip():
            continue
        key, separator, value = segment.partition('=')
        if not separator or not key.strip() or not value.strip():
            raise ValueError(f'{variable_name} contains an invalid segment.')
        settings[key.strip()] = value.strip()
    return settings

def get_setting(settings: dict[str, str], *names: str) -> str:
    by_name = {key.casefold(): value for key, value in settings.items()}
    for name in names:
        if value := by_name.get(name.casefold()):
            return value
    raise ValueError(f'Missing connection-string setting: {" or ".join(names)}')

azure_openai = parse_connection_string('AZURE_OPENAI_CONNECTION_STRING')
web_iq_mcp = parse_connection_string('WEB_IQ_MCP_CONNECTION_STRING')

azure_openai_endpoint = get_setting(azure_openai, 'Endpoint').rstrip('/')
azure_openai_model = get_setting(azure_openai, 'Deployment', 'Model')
mcp_server_url = get_setting(web_iq_mcp, 'Endpoint').rstrip('/')
mcp_headers = {
    key: value
    for key, value in web_iq_mcp.items()
    if key.casefold() != 'endpoint'
}

for name, url in {
    'Azure OpenAI endpoint': azure_openai_endpoint,
    'Web IQ MCP endpoint': mcp_server_url,
}.items():
    if urlparse(url).scheme != 'https':
        raise ValueError(f'{name} must use HTTPS: {url}')

required_mcp_headers = {'ocp-apim-subscription-key', 'x-apikey'}
actual_mcp_headers = {name.casefold() for name in mcp_headers}
if missing := required_mcp_headers - actual_mcp_headers:
    raise ValueError(f'Missing Web IQ MCP header setting(s): {sorted(missing)}')

print(f'Azure OpenAI endpoint: {azure_openai_endpoint}')
print(f'Azure OpenAI deployment: {azure_openai_model}')
print(f'Web IQ MCP endpoint: {mcp_server_url}')
print(f'MCP header names: {sorted(mcp_headers)}')


Azure OpenAI endpoint: https://jacwang-1123-resource.openai.azure.com
Azure OpenAI deployment: <deployment-name>
Web IQ MCP endpoint: https://apim-yvkpyuphlbctu.azure-api.net/web-iq/mcp
MCP header names: ['Ocp-Apim-Subscription-Key', 'x-apikey']


## 2. Create the Responses API client

Azure OpenAI's v1 API uses the standard `OpenAI` client. Normalizing the resource endpoint to `/openai/v1/` and acquiring the `https://ai.azure.com/.default` token matches the Microsoft Foundry Entra ID example. No Azure OpenAI API key is used.


In [8]:
from azure.identity import DefaultAzureCredential
from openai import APIStatusError, OpenAI
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AccessToken
import time

azure_openai_base_url = azure_openai_endpoint
if not azure_openai_base_url.endswith('/openai/v1'):
    azure_openai_base_url += '/openai/v1'




# Create a credential instance (can also use ClientSecretCredential, etc.)
credential = DefaultAzureCredential()

# Define the scope for your Azure resource
SCOPE = 'https://ai.azure.com/.default' #"https://cognitiveservices.azure.com/.default"

def my_bearer_token_provider() -> str:
    """
    Returns a fresh bearer token string for use in Authorization headers.
    """
    try:
        token: AccessToken = credential.get_token(SCOPE)
        return token.token
    except Exception as e:
        raise RuntimeError(f"Failed to acquire token: {e}")


azure_openai_token_provider = my_bearer_token_provider()

client = OpenAI(
    base_url=f'{azure_openai_base_url}/',
    api_key=azure_openai_token_provider,
)

print(f'Responses API base URL: {client.base_url}')


Responses API base URL: https://jacwang-1123-resource.openai.azure.com/openai/v1/


## 3. Invoke Web IQ through remote MCP

This is the Microsoft Foundry remote-MCP request shape: `type`, `server_label`, `server_url`, `headers`, and `require_approval`. `allowed_tools` limits discovery and model choice to Web IQ's `web` tool. Approval is disabled only for this controlled test.


In [9]:
web_iq_tool = {
    'type': 'mcp',
    'server_label': 'web_iq',
    'server_url': mcp_server_url,
    'headers': mcp_headers,
    'allowed_tools': ['web'],
    'require_approval': 'never',
}

prompt = (
    'Use the Web IQ web tool to find the current Microsoft documentation for '
    'Azure API Management AI gateway capabilities. Summarize three capabilities '
    'and include source URLs.'
)

response = client.responses.create(
    model="gpt-5.4-nano",
    tools=[web_iq_tool],
    tool_choice='required',
    input=prompt,
)

print(response.output_text)


Here are **three current Azure API Management AI gateway capabilities** from Microsoft documentation, with source URLs:

1) **Traffic mediation & model/tool endpoint management (multi-provider)**
- The AI gateway can manage different AI backends, including **OpenAI-compatible language model APIs**, **Anthropic Messages API**, **Google Vertex AI**, plus **remote MCP servers** and **A2A agent APIs**.  
Source: https://learn.microsoft.com/en-us/azure/api-management/genai-gateway-capabilities

2) **Token-based control for scalability and cost governance**
- Supports **LLM token rate limiting and quotas** (TPM/token counters) to prevent one consumer from exhausting shared capacity. Includes the ability to **pre-calculate prompt tokens** on the APIM side to reduce unnecessary calls to the model backend.  
Source: https://learn.microsoft.com/en-us/azure/api-management/genai-gateway-capabilities

3) **Security & safety controls for AI requests**
- Provides AI gateway security features such as 

## 4. Inspect and validate the MCP call

A successful response normally contains `mcp_list_tools` and `mcp_call` output items. The assertions turn the example into a small integration test without dumping large tool payloads.

The table below extracts `contentSafety` from each `mcp_call.output` and displays **Hate**, **Sexual**, **Violence**, and **SelfHarm** scores, scan status, chunks, and latency. This uses the gateway’s actual tool result; the model’s final prose may omit the telemetry. Scores are 0 (safe), 2 (low), 4 (medium), or 6 (high). A missing value is not a zero score. Rerun the tool invocation after updating the gateway; previously saved responses do not gain new telemetry. A `502` with `ContentSafetyAnalysisFailed` means moderation could not complete.


In [ ]:
import pandas as pd
from content_safety import extract_content_safety, content_safety_row

mcp_events = [item for item in response.output if item.type.startswith('mcp_')]
for item in mcp_events:
    print({
        'type': item.type,
        'server_label': getattr(item, 'server_label', None),
        'name': getattr(item, 'name', None),
        'error': getattr(item, 'error', None),
    })

web_calls = [
    item for item in mcp_events
    if item.type == 'mcp_call' and getattr(item, 'name', None) == 'web'
]
assert web_calls, 'The model did not call the Web IQ web tool.'
assert all(not getattr(item, 'error', None) for item in web_calls), 'The Web IQ web call failed.'
assert response.output_text.strip(), 'The model returned no final answer.'
print('PASS: Azure OpenAI called the Web IQ web tool through APIM.')

# Read the actual MCP tool output, which contains gateway moderation telemetry.
content_safety_results = pd.DataFrame([
    {
        'tool': call.name,
        **content_safety_row(extract_content_safety(getattr(call, 'output', None))),
    }
    for call in web_calls
])
if (content_safety_results['safety_status'] == 'missing').any():
    print('No Content Safety result in this saved response. Rerun the tool invocation with the updated gateway policy.')
content_safety_results


## 5. Verify that APIM blocks `browse`

This negative test exposes only `browse` to the model. The expected outcome is either an HTTP error returned by the Responses API or an `mcp_call` item containing an error after APIM returns `403 Forbidden`.


In [11]:
browse_tool = {
    **web_iq_tool,
    'allowed_tools': ['browse'],
}

try:
    blocked_response = client.responses.create(
        model=azure_openai_model,
        tools=[browse_tool],
        tool_choice='required',
        input='Use browse to retrieve https://news.microsoft.com/source/.',
    )
except APIStatusError as error:
    print(f'Caught APIStatusError: {error}')
    assert error.status_code >= 400
    print(f'PASS: browse was blocked; Responses API returned HTTP {error.status_code}.')
else:
    browse_calls = [
        item for item in blocked_response.output
        if item.type == 'mcp_call' and getattr(item, 'name', None) == 'browse'
    ]
    assert browse_calls, 'The model did not attempt the browse tool.'
    assert any(getattr(item, 'error', None) for item in browse_calls), (
        'Browse unexpectedly succeeded; verify that the APIM blocking policy is deployed.'
    )
    print('PASS: APIM rejected the MCP browse call.')


Caught APIStatusError: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
PASS: browse was blocked; Responses API returned HTTP 404.


## References

- [Use the Azure OpenAI Responses API: remote MCP servers](https://learn.microsoft.com/azure/foundry/openai/how-to/responses#using-remote-mcp-servers)
- [Microsoft Web IQ MCP](https://webiq.microsoft.ai/documentation/mcp/)
- [Web IQ overview](https://webiq.microsoft.ai/documentation/overview/)
